# Tutorial 8 — The Rhino MCP: AI-Assisted Data Engineering

This notebook is a **reference guide and prompt library** for reproducing the
Tutorials 1–4 workflow using Claude Desktop and the Rhino MCP Server.

Unlike the earlier tutorials, there is no code to run here. Instead, each
section shows:
1. The natural language prompt you type into Claude Desktop
2. Which Rhino MCP tool(s) Claude will call
3. What to expect in the response

**Prerequisites:**
- Complete the MCP setup in this tutorial's README
- Claude Desktop is open and the Rhino MCP server is connected
  (you should see a hammer icon in the chat interface)

---
## Step 1 — Connect and Verify Access

### What to type

```
What Rhino projects do I have access to?
```

### What Claude does
Calls `rhino_login` (auto-login from environment variables) then `rhino_project_list`.

### What you'll see
A list of your FCP projects with names and UIDs. Identify your project name —
you'll reference it by name in subsequent prompts.

---
## Step 2 — Register Datasets (Tutorial 1 via MCP)

### What to type

```
Register three datasets in my project with UID "<YOUR PROJECT UID>":

1. Name: "Patients — Site A"
   CSV path: /rhino_data/external/s3/intro_to_data_engineering/patients.csv

2. Name: "Encounters — Site A"
   CSV path: /rhino_data/external/s3/intro_to_data_engineering/encounters.csv

3. Name: "Procedures — Site A"
   CSV path: /rhino_data/external/s3/intro_to_data_engineering/procedures.csv

All datasets are de-identified.
```

### What Claude does
Calls `rhino_dataset_create` three times, once per dataset.

### What you'll see
Confirmation for each dataset with its UID. Claude will typically
summarize all three in a single response.

### Follow-up: Generate Schemas

```
Auto-generate data schemas for all three datasets I just registered.
For the encounters dataset, make sure DateOfService is typed as a date.
For the procedures dataset, make sure ProcedureDate is typed as a date.
```

### What Claude does
Calls `rhino_data_schema_create` for each dataset using schema auto-generation,
then updates the date column types as instructed.

### What you'll see
A table or list showing each schema's columns and inferred types,
with confirmation that date corrections were applied.

---
## Step 3 — Data Discovery (Tutorial 2 via MCP)

### Row counts

```
How many rows are in each of my three datasets?
```

### What Claude does
Calls `rhino_metric_run` with `Count()` for each dataset.

### Demographics

```
Show me the gender, race, and ethnicity distributions in the patients dataset.
```

### What Claude does
Calls `rhino_metric_run` with `Histogram` for each of the three columns.
Presents results as a formatted breakdown.

### Pro tip
Ask a follow-up to flag harmonization prep:
```
Which exact gender values appear in the data? I need to know the precise strings
so I can set up OMOP vocabulary mappings.
```

### Procedure codes

```
What CPT codes appear in the procedures dataset, and how often does each appear?
```

### What Claude does
Calls `rhino_metric_run` with `Histogram("ProcedureCode")`.

### Full dataset profile

```
Give me a full profile of the encounters dataset — row count, null rates on all
columns, and a breakdown of TypeOfService values.
```

### What Claude does
Calls `rhino_dataset_profile` which computes a comprehensive Table 1-style
summary in a single call.

---
## Step 4 — Data Preparation (Tutorial 3 via MCP)

### Create the code object

```
Create a Python code object in my "<YOUR PROJECT NAME>" project called
"Data Prep — All Datasets" that does the following to three input datasets
(patients at slot 0, encounters at slot 1, procedures at slot 2):

For all datasets:
- Standardize column names to snake_case
- Drop exact duplicate rows
- Drop rows missing required ID columns

For patients (slot 0):
- Standardize Gender values to title case (e.g. "male" → "Male")
- Validate that YearOfBirth is between 1900 and 2025
- Output to slot 0

For encounters (slot 1):
- Parse DateOfService to YYYY-MM-DD format
- Standardize TypeOfService to title case
- Drop rows missing patientID or visitID
- Output to slot 1

For procedures (slot 2):
- Parse ProcedureDate to YYYY-MM-DD format
- Drop rows where ProcedureCode is null
- Drop rows missing patientID or visitID
- Output to slot 2
```

### What Claude does
Generates the Python script in-context and calls `rhino_code_object_create`
with `type=PythonCode` and three input/output schema slots.

Claude will show you the code for review before confirming creation.

### Run the code object

```
Run the "Data Prep — All Datasets" code object on:
- Slot 0: Patients — Site A
- Slot 1: Encounters — Site A
- Slot 2: Procedures — Site A

Name the outputs:
- "Patients — Site A — Prepared"
- "Encounters — Site A — Prepared"
- "Procedures — Site A — Prepared"
```

### What Claude does
Calls `rhino_code_run` with the three dataset UIDs in slot order.

### Check status

```
Is my data prep run finished?
```

### What Claude does
Calls `rhino_run_get_status` with the run UID from the previous response.

---
## Step 5 — Harmonization (Tutorial 4 via MCP)

### Analyze source-to-target alignment

```
Analyze the "Patients — Site A — Prepared" dataset against my OMOP Person
target schema. Which columns need vocabulary mapping, and what vocabularies
are available in my project?
```

### What Claude does
Calls `rhino_harmonization` with `action="analyze"`. Returns a column alignment
table and lists available vocabularies with their UIDs.

### Setup all three mappings

```
Set up harmonization mappings for all three prepared datasets to OMOP:

1. "Patients — Site A — Prepared" → OMOP Person (table name: person)
   Semantic mappings needed:
   - gender column → Gender vocabulary
   - race column → Race vocabulary
   - ethnicity column → Ethnicity vocabulary

2. "Encounters — Site A — Prepared" → OMOP Visit Occurrence (table name: visit_occurrence)
   Semantic mappings needed:
   - type_of_service column → Visit Type vocabulary

3. "Procedures — Site A — Prepared" → OMOP Procedure Occurrence (table name: procedure_occurrence)
   Semantic mappings needed:
   - procedure_code column → CPT4 vocabulary
```

### What Claude does
Calls `rhino_harmonization` with `action="setup_mapping"` three times.
Creates semantic mapping objects and syntactic mapping skeletons for each pair.

### Review proposed term matches

```
Show me the proposed term matches for the gender semantic mapping.
```

### What Claude does
Calls `rhino_semantic_mapping` with `action="get_data"`.
Returns a table of source values and their AI-proposed OMOP matches.

### Approve all semantic mappings

```
Approve the following mappings for the gender semantic mapping:
- "Male" → OMOP concept 8507 (Male) — approved
- "Female" → OMOP concept 8532 (Female) — approved
- "Other" → OMOP concept 8521 (Other) — approved
```

```
Approve the following for the race semantic mapping:
- "Asian" → OMOP concept 8515 — approved
- "Black" → OMOP concept 8516 (Black or African American) — approved
- "White" → OMOP concept 8527 — approved
- "Other" → OMOP concept 8522 (Other Race) — approved
```

```
Approve the ethnicity mapping:
- "Hispanic" → OMOP concept 38003563 (Hispanic or Latino) — approved
- "Non-Hispanic" → OMOP concept 38003564 (Not Hispanic or Latino) — approved
```

```
Approve the visit type mapping:
- "Outpatient" → OMOP concept 9202 — approved
- "Inpatient" → OMOP concept 9201 — approved
- "Emergency" → OMOP concept 9203 — approved
```

```
Approve the CPT code mapping:
- CPT 45378 → OMOP concept 4287782 (Colonoscopy) — approved
- CPT 44950 → OMOP concept 4196867 (Appendectomy) — approved
- CPT 99203 → OMOP concept 4098498 — approved
- CPT 99212 → OMOP concept 4098460 — approved
- CPT 99213 → OMOP concept 4098462 — approved
- CPT 99285 → OMOP concept 4129922 — approved
```

### What Claude does (for each approval prompt)
Calls `rhino_semantic_mapping` with `action="approve"` and the entries you specified.

### Auto-generate syntactic mappings

```
Auto-generate the syntactic mapping field configurations for all three
harmonization mappings (Person, Visit Occurrence, Procedure Occurrence).
```

### What Claude does
Calls `rhino_syntactic_mapping` with `action="auto_generate"` for each mapping.

### Run harmonization

```
Run the harmonization for all three mappings.
```

### What Claude does
Calls `rhino_syntactic_mapping` with `action="run"` for each mapping,
passing the appropriate semantic mapping UIDs by vocabulary name.

### Verify results

```
How many rows are in each of my harmonized OMOP output datasets?
Show me the gender_concept_id distribution in the OMOP Person output.
```

### What Claude does
Calls `rhino_metric_run` with `Count()` on each harmonized dataset,
then `Histogram("gender_concept_id")` on the Person table.

---
## Comparison: SDK vs. MCP

The table below summarizes how each tutorial step maps between the two approaches.

| Step | SDK (Notebooks) | MCP (Claude Desktop) |
|------|-----------------|----------------------|
| Register datasets | `DatasetCreateInput` + `session.dataset.add_dataset()` | "Register patients.csv as a dataset..." |
| Generate schema | `session.data_schema.generate_data_schema_from_dataset()` | "Auto-generate a schema for this dataset" |
| Profile data | `dataset.get_metric(Histogram(...))` | "Show me the gender distribution" |
| Create code object | `CodeObjectCreateInput` + `session.code_object.add_code_object()` | "Create a Python code object that..." |
| Run code object | `CodeObjectRunInput` + `session.code_object.run_code_object()` | "Run the prep code object on these datasets" |
| Analyze harmonization | `session.harmonization.analyze()` | "Analyze this dataset against the OMOP Person schema" |
| Setup mapping | `session.harmonization.setup_mapping()` | "Set up the harmonization mapping" |
| Approve terms | `session.semantic_mapping.approve()` | "Approve Male → 8507, Female → 8532..." |
| Auto-generate syntactic | `session.syntactic_mapping.auto_generate()` | "Auto-generate the syntactic mapping" |
| Run harmonization | `session.syntactic_mapping.run()` | "Run the harmonization" |

Both approaches call the same underlying Rhino SDK and produce identical results.
The MCP is best for exploration, rapid iteration, and ad-hoc analysis.
The SDK notebooks are best for reproducible, version-controlled pipelines.

---
## Next Steps

You've now completed the full Rhino FCP data engineering workflow:

- ✅ Dataset registration and schema generation (Tutorial 1)
- ✅ Federated data discovery without accessing raw data (Tutorial 2)
- ✅ On-site data preparation via Python Code Objects (Tutorial 3)
- ✅ OMOP harmonization with the Rhino DHE (Tutorial 4)
- ✅ AI-assisted data engineering via the Rhino MCP (Tutorial 5)

The harmonized OMOP datasets are now ready for federated model training.
See the **Federated Learning workflow** for the next phase of the project.